
# 01 — Build shared AoU disease flags and ancestry PCs

This notebook creates shared resources exactly once for both omic analyses.

The old workflow repeated case extraction separately for proteins and chemistries. Here, the AoU-wide condition-hit table is queried once, saved, and then merged locally onto each omic cohort. This guarantees identical disease definitions and control exclusions.

**Outputs**

- `data/shared/aou_disease_concept_registry.tsv`
- `data/shared/aou_disease_flag_hits.tsv.gz`
- `data/shared/aou_genetic_ancestry_pcs.tsv`
- SQL snapshots in `logs/`

The concept registry records the concepts used for each case definition.


In [ ]:
from pathlib import Path
import os
import sys

# Jupyter's file-browser directory and the kernel working directory can differ.
# Locate the project by its marker instead of trusting Path.cwd().
def find_project_root():
    marker_name = ".aou_combined_replication"
    project_name = "Combined_VAE_linear_AoU_replication"

    env_root = os.getenv("AOU_COMBINED_REPLICATION_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / marker_name).exists():
            return root
        raise FileNotFoundError(
            f"AOU_COMBINED_REPLICATION_ROOT={root} lacks {marker_name}"
        )

    here = Path.cwd().resolve()
    ancestors = [here, *here.parents]
    for candidate in ancestors:
        if (candidate / marker_name).exists():
            return candidate

    matches = []
    for parent in ancestors[:5]:
        patterns = [
            f"*/{marker_name}",
            f"{project_name}/{marker_name}",
            f"*/{project_name}/{marker_name}",
        ]
        for pattern in patterns:
            matches.extend(path.parent for path in parent.glob(pattern))

    matches = sorted(set(path.resolve() for path in matches))
    named = [path for path in matches if path.name == project_name]
    matches = named or matches
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(
        "Could not locate the combined replication project. Set "
        "AOU_COMBINED_REPLICATION_ROOT explicitly. Candidates found: "
        + str(matches)
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import project_config as cfg
from src import aou_replication_utils as aru

print("Kernel working directory:", Path.cwd().resolve())
print("Project root:", cfg.PROJECT_ROOT)
print("Workspace root:", cfg.WORKSPACE_ROOT)


In [ ]:

import pandas as pd
from IPython.display import display
from google.cloud import bigquery

client = bigquery.Client(project=cfg.GOOGLE_PROJECT)
disease_defs = aru.load_disease_definitions(
    cfg.CONFIG_DIR / "disease_definitions.json"
)
base_domains = [item["domain"] for item in disease_defs]
patterns_cte = aru.build_disease_patterns_cte(disease_defs)

concept_query = aru.disease_concept_registry_query(cfg.CDR, patterns_cte)
flags_query = aru.disease_flag_hits_query(cfg.CDR, patterns_cte, disease_defs)

(cfg.LOG_DIR / "disease_concept_registry.sql").write_text(
    concept_query, encoding="utf-8"
)
(cfg.LOG_DIR / "disease_flag_hits.sql").write_text(
    flags_query, encoding="utf-8"
)

print("BigQuery CDR:", cfg.CDR)
print("Disease domains:", base_domains)


In [ ]:
from google.cloud import bigquery

print(cfg.bigquery_configuration_summary())


client = bigquery.Client(
    project=cfg.GOOGLE_PROJECT
)

probe_sql = f"""
SELECT
    concept_id,
    concept_name
FROM `{cfg.CDR}.concept`
LIMIT 3
"""

probe = client.query(
    probe_sql
).to_dataframe()

display(probe)

In [ ]:

concept_out = cfg.SHARED_DIR / "aou_disease_concept_registry.tsv"
flags_out = cfg.SHARED_DIR / "aou_disease_flag_hits.tsv.gz"

if concept_out.exists() and not cfg.FORCE_REBUILD:
    concept_registry = pd.read_csv(concept_out, sep="	")
    print("Reused cached concept registry:", concept_out)
else:
    concept_registry = client.query(concept_query).to_dataframe()
    concept_registry.to_csv(concept_out, sep="	", index=False)
    print("Queried and saved concept registry:", concept_out)

if flags_out.exists() and not cfg.FORCE_REBUILD:
    flag_hits = pd.read_csv(flags_out, sep="	", compression="gzip")
    print("Reused cached condition-hit flags:", flags_out)
else:
    flag_hits = client.query(flags_query).to_dataframe()
    flag_hits.to_csv(flags_out, sep="	", index=False, compression="gzip")
    print("Queried and saved condition-hit flags:", flags_out)

flag_hits["person_id"] = pd.to_numeric(
    flag_hits["person_id"], errors="raise"
).astype("int64")

print(f"Concept registry rows: {len(concept_registry):,}")
print(f"People with at least one disease/exclusion hit: {len(flag_hits):,}")
display(
    concept_registry.sort_values(
        ["disease_domain", "set_type", "n_people_all_aou"],
        ascending=[True, True, False],
    ).groupby(["disease_domain", "set_type"]).head(12)
)


In [ ]:

ancestry_source = cfg.first_existing(cfg.ANCESTRY_FILE_CANDIDATES)
pcs_out = cfg.SHARED_DIR / "aou_genetic_ancestry_pcs.tsv"

if pcs_out.exists() and not cfg.FORCE_REBUILD:
    pcs = pd.read_csv(pcs_out, sep="	")
    print("Reused cached PC table:", pcs_out)
else:
    pcs = aru.parse_ancestry_pcs(
        ancestry_source,
        n_pcs=cfg.N_GENETIC_PCS,
    )
    pcs.to_csv(pcs_out, sep="	", index=False)
    print("Parsed and saved PC table:", pcs_out)

if pcs["person_id"].duplicated().any():
    raise ValueError("PC table contains duplicate person_id values")

print(f"People with parsed PCs: {len(pcs):,}")
pc_complete = pcs[cfg.PC_COLUMNS].notna().all(axis=1)
print(f"Complete PC1-PC{cfg.N_GENETIC_PCS}: {pc_complete.sum():,}")
if "ancestry_pred" in pcs.columns:
    display(pcs["ancestry_pred"].fillna("Missing").value_counts())


In [ ]:

# Save an easily inspected disease-domain count table from the sparse hit file.
count_rows = []
for domain in base_domains:
    for set_type in ["case", "control_exclude"]:
        column = f"{domain}_{set_type}"
        count_rows.append(
            {
                "disease_domain": domain,
                "set_type": set_type,
                "n_people_with_hit": int(
                    pd.to_numeric(flag_hits.get(column, 0), errors="coerce")
                    .fillna(0)
                    .eq(1)
                    .sum()
                ),
            }
        )
hit_summary = pd.DataFrame(count_rows)
hit_summary.to_csv(
    cfg.SHARED_DIR / "aou_disease_flag_hit_summary.tsv",
    sep="	",
    index=False,
)
display(hit_summary)
